# Lab3 - Train Classifier (English Only)

Classifier training setup:
- Train languages: ['en']
- Features: early fusion (XLM-R text + PySceneDetect keyframe stats + sensorial)
- Target: majority label from `labels_task3_1` (`YES`/`NO`)


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
import joblib


In [ ]:
# Cluster configuration copied from run.sh / run_parallel.sh
SLURM_PARTITION = 'long'
SLURM_CPUS_PER_TASK = 8
SLURM_MEM = '32G'
SLURM_GRES_SEQUENTIAL = 'shard:6'  # run.sh
SLURM_GRES_PARALLEL = 'shard:4'    # run_parallel.sh
CONDA_ENV = 'RFA2526pt'
PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = PYTORCH_CUDA_ALLOC_CONF

PROJECT_ROOT = Path('/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DELIVERABLES_DIR = PROJECT_ROOT / 'entregables'
PREDICCIONES_FINALES_DIR = PROJECT_ROOT / 'predicciones_finales'
WEIGHTS_DIR = PROJECT_ROOT / 'weights'

DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICCIONES_FINALES_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

FUSION_TRAIN_PATH = ARTIFACTS_DIR / 'fusion_features_train.npz'
FUSION_TEST_PATH = ARTIFACTS_DIR / 'fusion_features_test.npz'
META_TRAIN_PATH = ARTIFACTS_DIR / 'sample_metadata_train.csv'
if not META_TRAIN_PATH.exists():
    META_TRAIN_PATH = ARTIFACTS_DIR / 'sample_metadata.csv'
TEST_JSON_PATH = PROJECT_ROOT.parent / 'materials' / 'dataset_task3_exist2026' / 'test.json'

GROUP_ID = 'BeingChillingWeWillWin'
MODEL_TAG = 'XLMRSceneSensor_LR_EN'
TEST_CASE = 'EXIST2026'
TRAIN_LANGS = ['en']

print('Training langs:', TRAIN_LANGS)
print('Using train artifacts:', FUSION_TRAIN_PATH)
print('Using test artifacts:', FUSION_TEST_PATH)
print('Final prediction dir:', PREDICCIONES_FINALES_DIR)

In [ ]:
required_paths = [FUSION_TRAIN_PATH, FUSION_TEST_PATH, META_TRAIN_PATH]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Missing split artifacts. Run 01_build_multimodal_features.ipynb to generate train/test artifacts first: '
        + ', '.join(missing_paths)
    )

train_bundle = np.load(FUSION_TRAIN_PATH, allow_pickle=True)
X_train_all = train_bundle['X_fusion']
y_train_all = train_bundle['y']
langs_train_all = train_bundle['langs'].astype(str)
sample_ids_train_all = train_bundle['sample_ids'].astype(str)

test_bundle = np.load(FUSION_TEST_PATH, allow_pickle=True)
X_test_all = test_bundle['X_fusion']
langs_test_all = test_bundle['langs'].astype(str)
sample_ids_test_all = test_bundle['sample_ids'].astype(str)

meta = pd.read_csv(META_TRAIN_PATH)
meta['sample_id'] = meta['sample_id'].astype(str)

print('X_train_all shape:', X_train_all.shape)
print('X_test_all shape:', X_test_all.shape)
print('Train samples:', len(sample_ids_train_all), '| Test samples:', len(sample_ids_test_all))
print('Train language distribution:', dict(pd.Series(langs_train_all).value_counts()))
print('Test language distribution:', dict(pd.Series(langs_test_all).value_counts()))


In [ ]:
train_mask = np.isin(langs_train_all, TRAIN_LANGS) & (y_train_all >= 0)

X_train_pool = X_train_all[train_mask]
y_train_pool = y_train_all[train_mask]

if len(np.unique(y_train_pool)) < 2:
    raise RuntimeError('Training subset has <2 classes. Check labels and language filter.')

print('Training subset size:', X_train_pool.shape[0])
print('Class distribution:', dict(pd.Series(y_train_pool).value_counts()))


In [ ]:
# Internal validation for quick quality check.
X_tr, X_va, y_tr, y_va = train_test_split(
    X_train_pool,
    y_train_pool,
    test_size=0.2,
    random_state=42,
    stratify=y_train_pool,
)

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2500, class_weight='balanced', n_jobs=-1)),
])

clf.fit(X_tr, y_tr)
val_pred = clf.predict(X_va)

f1_macro = f1_score(y_va, val_pred, average='macro')
print('Validation f1_macro:', f1_macro)
print(classification_report(y_va, val_pred, digits=4))


In [ ]:
# Final fit on complete language-specific training subset.
final_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2500, class_weight='balanced', n_jobs=-1)),
])
final_clf.fit(X_train_pool, y_train_pool)

meta = meta.sort_values('sample_id').reset_index(drop=True)
if not np.array_equal(sample_ids_train_all.astype(str), meta['sample_id'].astype(str).to_numpy()):
    raise RuntimeError('sample_metadata_train.csv is not aligned with fusion_features_train.npz by sample_id.')

if not TEST_JSON_PATH.exists():
    raise FileNotFoundError(f'test.json not found: {TEST_JSON_PATH}')
with open(TEST_JSON_PATH, 'r', encoding='utf-8') as f:
    test_raw = json.load(f)
test_ids = {str(k) for k in test_raw.keys()}

lang_mask_test = np.isin(langs_test_all.astype(str), TRAIN_LANGS)
test_id_mask = np.array([sid in test_ids for sid in sample_ids_test_all.astype(str)], dtype=bool)
infer_mask = test_id_mask & lang_mask_test
if not infer_mask.any():
    raise RuntimeError('No matching rows from test.json found for TRAIN_LANGS in fusion_features_test.npz.')

infer_reason = 'split pipeline: fusion_features_test.npz + test.json + TRAIN_LANGS'
infer_ids = sample_ids_test_all[infer_mask]
X_infer = X_test_all[infer_mask]
pred_infer = final_clf.predict(X_infer)

label_map = {0: 'NO', 1: 'YES'}
pred_labels = [label_map[int(v)] for v in pred_infer]

print('Inference subset reason:', infer_reason)
print('Inference subset size:', int(infer_mask.sum()))

model_path = WEIGHTS_DIR / f'{GROUP_ID}_{MODEL_TAG}.joblib'
joblib.dump(final_clf, model_path)
print('Saved model:', model_path)


In [ ]:
output_rows = []
for sid, pred_label in zip(infer_ids, pred_labels):
    output_rows.append({
        'test_case': TEST_CASE,
        'id': str(sid),
        'value': pred_label,
    })

json_path = PREDICCIONES_FINALES_DIR / f'{GROUP_ID}_{MODEL_TAG}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(output_rows, f, ensure_ascii=False)

print('Saved prediction json (predicciones_finales):', json_path)
print('Rows:', len(output_rows))
output_rows[:3]

## Notes

- Inference is now strict: only rows where split contains TEST are used.
- Predictions are saved in predicciones_finales/ under notebooks_shiyi.
- If TEST rows are missing in metadata, execution stops with an explicit error.